# 21 Transformer完整架构 - 自注意力、多头、位置编码

本教程将深入讲解Transformer的每一个组件，带你从零理解这个改变AI世界的架构。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、Transformer 总览

### 为什么Transformer如此重要？

2017年论文《Attention Is All You Need》提出了Transformer，彻底改变了NLP（乃至整个AI）：

| 特性 | RNN/LSTM | Transformer |
|------|----------|-------------|
| 计算方式 | 顺序（一步步） | 并行（全部一起） |
| 长距离依赖 | 困难（即使LSTM也有限） | 直接（自注意力连接所有位置） |
| 训练速度 | 慢 | 快（可并行） |
| 可扩展性 | 有限 | 极强（GPT、BERT等） |

### Transformer的整体结构

```
输入序列                    输出序列
   │                           │
   ▼                           ▼
┌─────────┐              ┌─────────┐
│Embedding│              │Embedding│
│+位置编码 │              │+位置编码 │
└────┬────┘              └────┬────┘
     │                        │
┌────▼────┐     交叉     ┌────▼────┐
│Encoder  │──注意力──→  │Decoder  │
│  N层    │             │  N层    │
└─────────┘             └────┬────┘
                             │
                             ▼
                        ┌─────────┐
                        │Linear+  │
                        │Softmax  │
                        └─────────┘
```

In [ ]:
# Transformer架构可视化
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_xlim(0, 16)
ax.set_ylim(0, 18)
ax.axis('off')

box_style = dict(boxstyle='round,pad=0.5', facecolor='lightblue', edgecolor='blue', alpha=0.8)
decoder_box = dict(boxstyle='round,pad=0.5', facecolor='lightcoral', edgecolor='red', alpha=0.8)
attn_box = dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='orange', alpha=0.9)

ax.set_title('Transformer Complete Architecture', fontsize=18, fontweight='bold')

# ====== 左侧：编码器 ======
ax.text(3.5, 17, 'Inputs\n输入序列', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))

ax.text(3.5, 15.5, 'Embedding + Positional Encoding\n词嵌入 + 位置编码', ha='center', va='center', fontsize=10, bbox=box_style)
ax.annotate('', xy=(3.5, 14.8), xytext=(3.5, 16.2),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Encoder Blocks
for i in range(3):
    y_base = 13.5 - i * 3.5
    if i == 0:
        ax.text(3.5, y_base, f'Encoder\nLayer {i+1}', ha='center', va='center', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.8', facecolor='lightblue', edgecolor='blue', alpha=0.8))
        ax.annotate('', xy=(3.5, y_base+0.5), xytext=(3.5, 14.8),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
    else:
        ax.text(3.5, y_base, f'Encoder\nLayer {i+1}', ha='center', va='center', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.8', facecolor='lightblue', edgecolor='blue', alpha=0.8))
        ax.annotate('', xy=(3.5, y_base+0.5), xytext=(3.5, y_base+1.0),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # 内部结构
    ax.text(3.5, y_base+0.15, 'Multi-Head Self-Attention\n + Add & Norm', ha='center', va='center', fontsize=8,
            bbox=attn_box)
    ax.text(3.5, y_base-0.25, 'Feed Forward\n + Add & Norm', ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green', alpha=0.9))

# ====== 右侧：解码器 ======
ax.text(12.5, 17, 'Outputs (shifted right)\n输出序列(右移)', ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))

ax.text(12.5, 15.5, 'Embedding + Positional Encoding', ha='center', va='center', fontsize=10, bbox=decoder_box)
ax.annotate('', xy=(12.5, 14.8), xytext=(12.5, 16.2),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

for i in range(3):
    y_base = 13.5 - i * 3.5
    ax.text(12.5, y_base, f'Decoder\nLayer {i+1}', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.8', facecolor='lightcoral', edgecolor='red', alpha=0.8))
    
    ax.text(12.5, y_base+0.2, 'Masked Multi-Head\n Self-Attention + Add&Norm', ha='center', va='center', fontsize=8,
            bbox=attn_box)
    ax.text(12.5, y_base-0.15, 'Multi-Head Cross\n Attention + Add&Norm', ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lavender', edgecolor='purple', alpha=0.9))
    ax.text(12.5, y_base-0.4, 'Feed Forward\n + Add & Norm', ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green', alpha=0.9))

# ====== 连接线 ======
ax.annotate('', xy=(12.5, 14.8), xytext=(12.5, 16.2),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Encoder→Decoder 交叉注意力连接
ax.annotate('', xy=(8, 11.5), xytext=(5, 11.5),
           arrowprops=dict(arrowstyle='->', color='purple', lw=2.5, linestyle='--'))
ax.text(6.5, 12, 'K, V', fontsize=9, color='purple', style='italic')
ax.annotate('', xy=(11, 11.5), xytext=(8, 11.5),
           arrowprops=dict(arrowstyle='->', color='purple', lw=2.5, linestyle='--'))
ax.text(9.5, 12, 'Q', fontsize=9, color='purple', style='italic')

# ====== 输出 ======
ax.text(12.5, 1, 'Linear + Softmax\n线性变换 + 归一化', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='orange', alpha=0.8))
ax.annotate('', xy=(12.5, 1.7), xytext=(12.5, 2.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(12.5, 0, 'Output\nProbabilities\n输出概率分布', ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
ax.annotate('', xy=(12.5, 0.4), xytext=(12.5, 0.7),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Encoder→Decoder 跨层连接
ax.annotate('', xy=(10, 4.5), xytext=(5, 4.5),
           arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, linestyle='--'))
ax.text(7.5, 4.8, 'Encoder output as K,V for Cross-Attention', fontsize=8, color='purple', style='italic')

# N × 标记
ax.text(2, 8, 'N×', fontsize=16, ha='center', va='center', color='blue')
ax.text(10.5, 8, 'N×', fontsize=16, ha='center', va='center', color='red')

plt.tight_layout()
plt.show()

## 二、自注意力（Self-Attention）详解

### 什么是自注意力？

自注意力让序列中的**每个词都能关注到序列中的所有词**（包括自己），从而捕捉词与词之间的关系。

### 三个关键向量

对于每个词，我们计算三个向量：

- **Query (Q)**："我想找什么？"——当前词的查询向量
- **Key (K)**："我有什么？"——每个词的特征向量
- **Value (V)**："我的实际内容"——每个词的信息向量

### 计算步骤

1. **计算注意力分数**: $Score = Q \cdot K^T$
2. **缩放**: 除以 $\sqrt{d_k}$ 防止梯度消失
3. **Softmax**: 归一化为概率分布
4. **加权求和**: $Attention = \text{softmax}(Score) \cdot V$

In [ ]:
# 自注意力详细计算过程
print("=" * 70)
print("自注意力（Self-Attention）逐步计算")
print("=" * 70)
print()

# 示例句子
sentence = ["I", "love", "machine", "learning"]
print(f"输入句子: {sentence}")
print()

# 步骤0: 词嵌入 (简化为小维度)
torch.manual_seed(42)
d_model = 4  # 嵌入维度
n_tokens = len(sentence)

# 假设的词嵌入
embeddings = torch.randn(n_tokens, d_model)
print(f"Step 0: 词嵌入 (d_model={d_model})")
for i, word in enumerate(sentence):
    print(f"  {word:10s} → {embeddings[i].detach().numpy()}")
print()

# 步骤1: 计算 Q, K, V
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

Q = W_Q(embeddings)
K = W_K(embeddings)
V = W_V(embeddings)

print(f"Step 1: 计算 Q, K, V 矩阵")
print(f"  Q 形状: {Q.shape}, K 形状: {K.shape}, V 形状: {V.shape}")
print()

# 步骤2: 计算注意力分数
scores = Q @ K.T  # (n_tokens, n_tokens)
print(f"Step 2: 注意力分数 (Q·K^T)")
print(f"  分数矩阵形状: {scores.shape}")
print(f"  原始分数:")
print(f"    ", end="")
for word in sentence:
    print(f"{word:>12s}", end="")
print()
for i, word in enumerate(sentence):
    print(f"  {word:10s}", end="")
    for j in range(n_tokens):
        print(f"{scores[i,j].item():>12.3f}", end="")
    print()
print()

# 步骤3: 缩放 + Softmax
d_k = torch.tensor(d_model, dtype=torch.float32)
scaled_scores = scores / torch.sqrt(d_k)
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Step 3: 缩放 (÷√{d_k}={torch.sqrt(d_k).item():.2f}) + Softmax")
print(f"  注意力权重 (每行和为1):")
for i, word in enumerate(sentence):
    print(f"  {word:10s}: ", end="")
    for j in range(n_tokens):
        print(f"{attention_weights[i,j].item():.3f}  ", end="")
    print(f" (和={attention_weights[i].sum().item():.3f})")
print()

# 步骤4: 加权求和
output = attention_weights @ V
print(f"Step 4: 输出 = Attention权重 × V")
print(f"  输出形状: {output.shape}")
for i, word in enumerate(sentence):
    print(f"  {word:10s} → {output[i].detach().numpy()}")

In [ ]:
# 解释自注意力的输出
print("\n=== 自注意力的直观理解 ===")
print()

print("注意力权重矩阵的含义：")
print(f"  行: 查询词 (Query)")
print(f"  列: 被关注的词 (Key)")
print()
print(f"  attention_weights[i][j] = 词 '{sentence[i]}' 对词 '{sentence[j]}' 的关注程度")
print()

# 可视化注意力权重
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attention_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)

ax.set_xticks(range(n_tokens))
ax.set_xticklabels(sentence, fontsize=12)
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(sentence, fontsize=12)

ax.set_xlabel('Key Words', fontsize=12)
ax.set_ylabel('Query Words', fontsize=12)
ax.set_title('Self-Attention Weight Matrix', fontsize=14, fontweight='bold')

for i in range(n_tokens):
    for j in range(n_tokens):
        val = attention_weights[i, j].item()
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
               color='white' if val > 0.4 else 'black', fontsize=11)

fig.colorbar(im, ax=ax, label='注意力权重')
plt.tight_layout()
plt.show()

# 解释
print("解读：")
for i, word in enumerate(sentence):
    weights = attention_weights[i].detach().numpy()
    max_idx = weights.argmax()
    print(f"  '{word}' 最关注的词: '{sentence[max_idx]}' (权重={weights[max_idx]:.3f})")

## 三、位置编码（Positional Encoding）

### 为什么需要位置编码？

自注意力机制**不考虑位置信息**——如果把句子的词重新排列，自注意力的输出（不考虑顺序的话）不会改变。因为它只关心词之间的语义关系，不关心它们的顺序。

但顺序在语言中非常重要：
- "猫追狗" ≠ "狗追猫"
- "I don't like it" ≠ "I like it don't"

### 正弦/余弦位置编码

原始Transformer使用正弦和余弦函数来编码位置：

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

其中：
- $pos$ 是位置（0, 1, 2, ...）
- $i$ 是维度索引
- $d$ 是嵌入维度

### 为什么用正弦/余弦？

1. **每个位置有唯一的编码**
2. **相对位置可学习**：$PE_{pos+k}$ 可以表示为 $PE_{pos}$ 的线性函数
3. **可以外推到更长的序列**（推理时可以处理比训练时更长的序列）

In [ ]:
# 位置编码可视化
print("=" * 70)
print("位置编码详解")
print("=" * 70)
print()

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # 创建位置编码矩阵
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维度用sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维度用cos
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

d_model = 16
max_len = 50
pos_enc = PositionalEncoding(d_model, max_len)

# 可视化位置编码
pe_matrix = pos_enc.pe[:max_len, :].numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 完整的位置编码热力图
im = axes[0, 0].imshow(pe_matrix.T, cmap='coolwarm', aspect='auto')
axes[0, 0].set_xlabel('Position')
axes[0, 0].set_ylabel('Dimension')
axes[0, 0].set_title(f'Positional Encoding Heatmap (d_model={d_model})')
fig.colorbar(im, ax=axes[0, 0], label='Encoding Value')

# 2. 几个维度的位置编码曲线
dims_to_plot = [0, 1, 2, 3]
for dim in dims_to_plot:
    axes[0, 1].plot(range(max_len), pe_matrix[:max_len, dim], 
                   linewidth=1.5, label=f'维度 {dim}')
axes[0, 1].set_xlabel('Position')
axes[0, 1].set_ylabel('Encoding Value')
axes[0, 1].set_title('Positional Encoding Curves by Dimension')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 两个位置的编码对比
pos_0 = pe_matrix[0, :]
pos_10 = pe_matrix[10, :]
axes[1, 0].bar(np.arange(d_model) - 0.2, pos_0, width=0.4, label='位置 0', alpha=0.7)
axes[1, 0].bar(np.arange(d_model) + 0.2, pos_10, width=0.4, label='位置 10', alpha=0.7)
axes[1, 0].set_xlabel('Dimension')
axes[1, 0].set_ylabel('Encoding Value')
axes[1, 0].set_title('Encoding Comparison at Different Positions')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 位置编码的逐元素加法
word_embedding = np.random.randn(d_model) * 0.1
combined = word_embedding + pe_matrix[5, :]
x = np.arange(d_model)
axes[1, 1].bar(x - 0.25, word_embedding, width=0.2, label='词嵌入', alpha=0.7, color='blue')
axes[1, 1].bar(x, pe_matrix[5, :], width=0.2, label='位置编码(位置5)', alpha=0.7, color='orange')
axes[1, 1].bar(x + 0.25, combined, width=0.2, label='相加后', alpha=0.7, color='green')
axes[1, 1].set_xlabel('Dimension')
axes[1, 1].set_ylabel('Value')
axes[1, 1].set_title('Embedding + Positional Encoding')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印具体数值
print(f"位置编码的具体值 (d_model={d_model}):")
for pos in [0, 1, 2]:
    print(f"  位置 {pos}: {np.round(pe_matrix[pos, :6], 4)}... (显示前6维)")

## 四、Transformer的各层组件

现在我们把所有组件拼装成完整的Transformer。

In [ ]:
# 从零实现完整的Transformer
print("=" * 70)
print("完整Transformer实现")
print("=" * 70)
print()

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, Q_input, K_input, V_input, mask=None):
        batch_size = Q_input.size(0)
        
        # 线性变换
        Q = self.W_Q(Q_input).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K_input).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V_input).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        output = (attn_weights @ V)
        
        # 拼接+输出变换
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_O(output), attn_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-Norm: 先LayerNorm再做注意力
        attn_out, attn_weights = self.self_attn(x, x, x, mask)
        x = x + self.dropout(attn_out)  # 残差连接
        x = self.norm1(x)
        
        ff_out = self.feed_forward(x)
        x = x + self.dropout(ff_out)  # 残差连接
        x = self.norm2(x)
        
        return x, attn_weights

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = x + self.dropout(attn_out)
        x = self.norm1(x)
        
        # Cross-Attention (Q来自decoder, K,V来自encoder)
        cross_out, _ = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = x + self.dropout(cross_out)
        x = self.norm2(x)
        
        # Feed Forward
        ff_out = self.feed_forward(x)
        x = x + self.dropout(ff_out)
        x = self.norm3(x)
        
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, n_heads=4, 
                 n_layers=3, d_ff=128, max_len=100, dropout=0.1):
        super().__init__()
        
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # 编码
        src_embedded = self.dropout(self.pos_encoding(self.src_embedding(src)))
        enc_output = src_embedded
        for layer in self.encoder_layers:
            enc_output, _ = layer(enc_output, src_mask)
        
        # 解码
        tgt_embedded = self.dropout(self.pos_encoding(self.tgt_embedding(tgt)))
        dec_output = tgt_embedded
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)
        
        return self.fc_out(dec_output)

# 创建模型
src_vocab = 1000
tgt_vocab = 1000
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 128

model = Transformer(src_vocab, tgt_vocab, d_model, n_heads, n_layers, d_ff)

print(f"Transformer配置:")
print(f"  词表大小: 源={src_vocab}, 目标={tgt_vocab}")
print(f"  d_model (嵌入维度): {d_model}")
print(f"  n_heads (注意力头数): {n_heads}")
print(f"  n_layers (层数): {n_layers}")
print(f"  d_ff (前馈网络维度): {d_ff}")
print(f"  每头维度: d_model/n_heads = {d_model}/{n_heads} = {d_model//n_heads}")
print()
print(f"总参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"  嵌入层: {sum(p.numel() for p in model.src_embedding.parameters()) + sum(p.numel() for p in model.tgt_embedding.parameters()):,}")
print(f"  编码器: {sum(p.numel() for p in model.encoder_layers.parameters()):,}")
print(f"  解码器: {sum(p.numel() for p in model.decoder_layers.parameters()):,}")
print(f"  输出层: {sum(p.numel() for p in model.fc_out.parameters()):,}")

In [ ]:
# 测试Transformer的前向传播
print("\n=== Transformer前向传播测试 ===")
print()

batch_size = 2
src_len = 8
tgt_len = 10

# 创建随机输入
src = torch.randint(0, src_vocab, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab, (batch_size, tgt_len))

# 创建mask
src_mask = torch.ones(batch_size, 1, 1, src_len)  # 不mask任何内容
tgt_mask = torch.tril(torch.ones(batch_size, 1, tgt_len, tgt_len))  # 下三角mask

model.eval()
with torch.no_grad():
    output = model(src, tgt, src_mask, tgt_mask)

print(f"输入:")
print(f"  src shape: {src.shape} (batch={batch_size}, seq_len={src_len})")
print(f"  tgt shape: {tgt.shape} (batch={batch_size}, seq_len={tgt_len})")
print()
print(f"输出:")
print(f"  output shape: {output.shape} (batch={batch_size}, seq_len={tgt_len}, vocab={tgt_vocab})")
print()

# 检查输出
output_probs = F.softmax(output, dim=-1)
preds = output.argmax(dim=-1)

print(f"预测token (argmax):")
for b in range(batch_size):
    print(f"  样本{b}: {preds[b].tolist()}")

print(f"\n输出概率范围: [{output_probs.min().item():.6f}, {output_probs.max().item():.6f}]")
print(f"每行概率和: {output_probs.sum(dim=-1)}"  # 应该是1
      )

In [ ]:
# 残差连接和LayerNorm的重要性
print("\n=== 残差连接和LayerNorm的作用 ===")
print()

# 对比有/无残差连接的梯度流
torch.manual_seed(42)

class WithResidual(nn.Module):
    def __init__(self, dim, n_layers):
        super().__init__()
        self.layers = nn.ModuleList([nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.ReLU()
        ) for _ in range(n_layers)])
    
    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)  # 残差连接
        return x

class WithoutResidual(nn.Module):
    def __init__(self, dim, n_layers):
        super().__init__()
        self.layers = nn.Sequential(*[
            nn.Sequential(nn.Linear(dim, dim), nn.ReLU())
            for _ in range(n_layers)
        ])
    
    def forward(self, x):
        return self.layers(x)

dim = 64
depths = [2, 4, 8, 16, 32]

print(f"{'深度':<8} {'有残差 梯度范数':>18} {'无残差 梯度范数':>18}")
print("-" * 50)

for depth in depths:
    # 有残差
    model_with = WithResidual(dim, depth)
    x = torch.randn(1, dim, requires_grad=True)
    output = model_with(x)
    loss = output.sum()
    loss.backward()
    grad_norm_with = x.grad.norm().item()
    
    # 无残差
    model_without = WithoutResidual(dim, depth)
    x2 = torch.randn(1, dim, requires_grad=True)
    output2 = model_without(x2)
    loss2 = output2.sum()
    loss2.backward()
    grad_norm_without = x2.grad.norm().item()
    
    print(f"{depth:<8} {grad_norm_with:>18.6f} {grad_norm_without:>18.6f}")

print("\n结论: 无残差连接时，深层网络的输入梯度会迅速衰减（梯度消失）")
print("     残差连接让梯度可以直接流过任意层数")

## 五、本章总结

### Transformer的核心组件

| 组件 | 作用 | 关键公式 |
|------|------|----------|
| **自注意力** | 让每个词关注所有其他词 | $\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$ |
| **多头注意力** | 多角度关注信息 | $\text{Concat}(head_1,...,head_h)W^O$ |
| **位置编码** | 注入序列位置信息 | $\sin(pos/10000^{2i/d}), \cos(...)$ |
| **残差连接** | 防止梯度消失 | $x + \text{Sublayer}(x)$ |
| **LayerNorm** | 稳定训练 | 对每个token的特征归一化 |
| **Feed Forward** | 非线性变换 | $\text{ReLU}(xW_1 + b_1)W_2 + b_2$ |

### Pre-Norm vs Post-Norm

本教程使用**Pre-Norm**架构（在子层之前做LayerNorm），这是现代Transformer的标准做法：
- 更稳定，适合深层网络
- 原始论文用Post-Norm，后来发现Pre-Norm更好

### 下一篇预告

下一篇将**训练一个Transformer模型**来完成实际任务！